In [ ]:
import pandas as pd
from rapidfuzz import process, fuzz
import re

In [ ]:
# defining input and output files 
input="credit_txn_v5.xlsx"
output="output_file.xlsx"


In [ ]:
df=pd.read_excel(input)

In [ ]:
# normalising ledger name to ignore extra spacing , special characters and perform case-insensitive analysis

def normalize_ledger(name):
    name = str(name).lower()
    name = re.sub(r'[^a-z\s]', ' ', name)
    name = re.sub(r'(.)\1{2,}', r'\1', name)  # charrrge → charge
    name = re.sub(r'\s+', ' ', name).strip()
    return name

df['ledger_norm'] = df['Ledger Name'].apply(normalize_ledger)

In [ ]:
# counting freq of normalised ledger names 
freq_df = (
    df.groupby('ledger_norm')
      .size()
      .reset_index(name='frequency')
      .sort_values('frequency', ascending=False)
      .reset_index(drop=True)
)



In [ ]:
# finding unique normailsed ledger names
unique_ledgers = freq_df['ledger_norm'].values.tolist()


In [ ]:
# ignoring ledgers whose freq <5
MIN_FREQ = 5

category_df = freq_df[freq_df['frequency'] >= MIN_FREQ].copy()
category_list = category_df['ledger_norm'].tolist()
category_freq = dict(zip(category_df['ledger_norm'], category_df['frequency']))

In [ ]:
# using rapidfuzz to check similar normalised ledger names
# putting them into same level 1 category and category name will be normalised ledger name with highest frequency in that category

SIM_THRESHOLD = 90

def match_category(ledger):
    match = process.extractOne(
        ledger,
        category_list,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=SIM_THRESHOLD
    )
    return match[0] if match else "OTHER"

ledger_to_category = {}

for i, ledger in enumerate(unique_ledgers):
    ledger_to_category[ledger] = match_category(ledger)

    # progress indicator every 1000 ledgers
    if i % 1000 == 0:
        print(f"Processed {i}/{len(unique_ledgers)}")


df['Ledger Category'] = df['ledger_norm'].map(ledger_to_category)